In [ ]:
#| hide
from fastllm_claude_code.core import *

# fastllm-claude-code

> Claude Code API for fastllm

This FastLLM provider runs models through an authenticated Claude Code CLI using [fastclaude](https://github.com/AnswerDotAI/fastclaude). Ordinary streaming and non-streaming FastLLM calls work as with any provider. Client-owned tool loops do not: continuing after a tool call requires `previous_response_id`, because the paused turn lives in a running CLI process that a regular completions-style history replay cannot recreate.

## Usage

### Installation

Install latest from the GitHub [repository][repo]:

```sh
$ pip install git+https://github.com/AnswerDotAI/fastllm-claude-code.git
```

or from [conda][conda]

```sh
$ conda install -c AnswerDotAI fastllm_claude_code
```

or from [pypi][pypi]


```sh
$ pip install fastllm_claude_code
```


[repo]: https://github.com/AnswerDotAI/fastllm-claude-code
[docs]: https://AnswerDotAI.github.io/fastllm-claude-code/
[pypi]: https://pypi.org/project/fastllm-claude-code/
[conda]: https://anaconda.org/AnswerDotAI/fastllm-claude-code

### Documentation

Documentation can be found hosted on this GitHub [repository][repo]'s [pages][docs]. Additionally you can find package manager specific guidelines on [conda][conda] and [pypi][pypi] respectively.

[repo]: https://github.com/AnswerDotAI/fastllm-claude-code
[docs]: https://AnswerDotAI.github.io/fastllm-claude-code/
[pypi]: https://pypi.org/project/fastllm-claude-code/
[conda]: https://anaconda.org/AnswerDotAI/fastllm-claude-code

## How to use

Installing the package registers the `claude_code` transport through FastLLM's provider entry point. The Claude CLI must already be installed and authenticated for the current user.

A normal FastLLM call uses the provider prefix:

```python
from fastllm.acomplete import acomplete

answer = await acomplete('Answer briefly: what is 2+2?', model='claude_code/claude-sonnet-5')
```

For a client-owned tool loop, pass standard Responses API or Chat Completions function schemas. Continuing after a tool call requires `previous_response_id`: it is not a regular completions turn, since replaying history cannot resume the paused process. The parameter is FastLLM's standard continuation contract, shared with the Responses transport, but here the id names a paused local process rather than stored server state. A response containing tool calls has a `response_id` naming that process:

```python
first = await acomplete(messages, model='claude_code/claude-sonnet-5', tools=tools)
assert first.tool_calls and first.response_id
assert not first.response_id_reusable

final = await acomplete(tool_result_messages, model='claude_code/claude-sonnet-5',
    tools=tools, previous_response_id=first.response_id)
assert final.response_id is None
```

The continuation ID is opaque, one-shot, and process-local. It expires after `response_ttl` seconds (3600 by default), and `claude_cancel(response_id)` closes an abandoned paused process. Reusing an expired or consumed ID raises a 409 `invalid_previous_response_id` error. A terminal response deliberately has no ID; a later top-level turn should replay canonical history into a fresh Claude process.

Set `stream=True` for FastLLM's normalized async stream. Non-streaming calls collect the same stream into one `Completion`. The adapter returns tool requests but never executes them.